! pip install requests load_dotenev
! pip install 

In [1]:
import requests # Importing the requests library for making HTTP requests
import pandas as pd # Importing pandas for data manipulation and analysis
import json # Importing json for handling JSON data
import os # Importing os for operating system dependent functionality
from dotenv import load_dotenv # Importing load_dotenv to load environment variables from a .env file

In [2]:
load_dotenv()
# Loading environment variables from a .env file
# This allows us to use sensitive information like API keys without hardcoding them in the script.

# Fetching the Etherscan API key from environment variables
ETHERSCAN_API_KEY = os.getenv('ETHERSCAN_API_KEY') 

# Fetching the CoinPaprika API key from environment variables
COINGECKO_API_KEY = os.getenv('COINGECKO_API_KEY') 

if not ETHERSCAN_API_KEY or not COINGECKO_API_KEY:
    raise ValueError("Please set the ETHERSCAN_API_KEY and COINGECKO_API_KEY environment variables.")

In [3]:
etherscan_url = 'https://api.etherscan.io/v2/api' # Etherscan API endpoint we will be using

In [4]:
# We structure parameters as a dictionary to allow for easy expansion in the future
# If we need to add more parameters, we can simply add them to this dictionary

params = {
    'chainid':1, # Chain ID for Ethereum Mainnet
    'module':'account', # Module for account-related actions
    'action':'balance', # Action to get the account balance
    'address':'0xde0b295669a9fd93d5f28d9ec85e40f4cb697bae',
    'tag':'latest', # Tag to specify the latest block
    'apikey':ETHERSCAN_API_KEY # API Key
}

response = requests.get(url=etherscan_url, params=params) # Making a GET request to the Etherscan API with the specified parameters
response.raise_for_status() # Raises an HTTPError for bad responses (4xx or 5xx status codes)
data = response.json() # Parsing the JSON response from the API

In [ ]:
# Display the data
print(f'API Response:\n{json.dumps(data, indent=4)}') 

# Print the JSON data with an indentation of 4 spaces

API Response:
{
    "status": "1",
    "message": "OK",
    "result": "181774355015033119539577"
}


In [6]:
type(data) # Display the type of the data variable to confirm it's a dictionary

dict

In [7]:
data.keys() # Display the keys of the data dictionary to understand its structure

dict_keys(['status', 'message', 'result'])

In [ ]:
# Convert the balance from Wei to Ether
# Etherscan returns balances in Wei, which is the smallest unit of Ether
# 1 Ether = 10^18 Wei, so we divide the balance in Wei by 10^18 to get the balance in Ether

balance_wei = int(data['result']) # Extracting the balance in Wei from the response
balance_ether = balance_wei / 10**18 # Converting Wei to Ether
print(f'Balance in Ether: {balance_ether}') # Displaying the balance in Ether

Balance in Ether: 183774.35474609083


In [8]:
# Get Transaction status by Transaction Hash

params = {
    'chainid':1, # Chain ID for Ethereum Mainnet
    'module':'transaction', # Module for transaction-related actions
    'action':'gettxreceiptstatus', # Action to get the account balance
    'txhash':'0x4ea4d96a92ce1266f96d7bb60577f46c90eff421a242bff3d9ac64fbd00d0870',
    'apikey':ETHERSCAN_API_KEY # API Key
}

response = requests.get(url=etherscan_url, params=params) # Making a GET request to the Etherscan API with the specified parameters
response.raise_for_status() # Raises an HTTPError for bad responses (4xx or 5xx status codes)
data = response.json() # Parsing the JSON response from the API

In [9]:
# Display the data
print(f'API Response:\n{json.dumps(data, indent=4)}') 

API Response:
{
    "status": "1",
    "message": "OK",
    "result": {
        "status": ""
    }
}


### Get "Normal" transactions for a specific address

In [ ]:
# Normal transactions are those that are not contract interactions or internal transactions
# They are typically user-initiated transactions that transfer Ether or tokens between addresses.

# This returns a max of 1000 transactions per request, so if you need more, you will have to paginate through the results.

params = {
    'chainid':1, # Chain ID for Ethereum Mainnet
    'module':'account', # Module for account-related actions
    'action':'txlist', # Action to get the transaction list
    'address':'0xde0b295669a9fd93d5f28d9ec85e40f4cb697bae', # Address to get transactions for
    'startblock':0, # Starting block number (0 for the earliest block)
    'endblock':99999999, # Ending block number (99999999 for the latest block)
    'sort':'asc', # Sort order (asc for ascending)
    'apikey':ETHERSCAN_API_KEY # API Key
}

response = requests.get(url=etherscan_url, params=params) # Making a GET request to the Etherscan API with the specified parameters
response.raise_for_status() # Raises an HTTPError for bad responses (4xx or 5xx status codes)
data = response.json() # Parsing the JSON response from the API

# Display the data
print(f'API Response:\n{json.dumps(data, indent=4)}')



API Response:
{
    "status": "1",
    "message": "OK",
    "result": [
        {
            "blockNumber": "54092",
            "timeStamp": "1439048640",
            "hash": "0x9c81f44c29ff0226f835cd0a8a2f2a7eca6db52a711f8211b566fd15d3e0e8d4",
            "nonce": "0",
            "blockHash": "0xd3cabad6adab0b52eb632c386ea194036805713682c62cb589b5abcd76de2159",
            "transactionIndex": "0",
            "from": "0x5abfec25f74cd88437631a7731906932776356f9",
            "to": "",
            "value": "11901464239480000000000000",
            "gas": "2000000",
            "gasPrice": "10000000000000",
            "isError": "0",
            "txreceipt_status": "",
            "input": "0x6060604052600760018181557305096a47749d8bfab0a90c1bb7a95115dbe4cea6600355737c56d94ebeccb769524379c450873519a9d805ff600490815573cda0ad7542e30bf520652a05056ebe0105c7e49a60055573775e18be7a50a0abb8a4e82b1bd697d79f31fe0460065573063dd253c8da4ea9b12105781c9611b8297f5d1490925573036c8cecce8d8bbf0831d840d7

In [11]:
# Let's convert the transactions from the API response into a pandas DataFrame for easier manipulation and analysis

transactions = pd.DataFrame(data['result']) # Converting the transactions to a pandas DataFrame
transactions['timeStamp'] = pd.to_datetime(transactions['timeStamp'], unit='s') # Converting the timestamp from Unix time to a human-readable format

transactions.head()

C:\Users\PC\AppData\Local\Temp\ipykernel_18320\3219569300.py:4: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  transactions['timeStamp'] = pd.to_datetime(transactions['timeStamp'], unit='s') # Converting the timestamp from Unix time to a human-readable format


,blockNumber,timeStamp,hash,nonce,blockHash,transactionIndex,from,to,value,gas,gasPrice,isError,txreceipt_status,input,contractAddress,cumulativeGasUsed,gasUsed,confirmations,methodId,functionName
0,54092,2015-08-08 15:44:00,0x9c81f44c29ff0226f835cd0a8a2f2a7eca6db52a711f...,0,0xd3cabad6adab0b52eb632c386ea194036805713682c6...,0,0x5abfec25f74cd88437631a7731906932776356f9,,11901464239480000000000000,2000000,10000000000000,0,,0x6060604052600760018181557305096a47749d8bfab0...,0xde0b295669a9fd93d5f28d9ec85e40f4cb697bae,1436963,1436963,23044078,0x60606040,
1,65204,2015-08-10 18:54:49,0x98beb27135aa0a25650557005ad962919d6a278c4b3d...,0,0x373d339e45a701447367d7b9c7cef84aab79c2b27142...,0,0x3fb1cd2cd96c6d5c0b5eb3322d807b34482481d4,0xde0b295669a9fd93d5f28d9ec85e40f4cb697bae,0,122261,50000000000,0,,0xf00d4b5d000000000000000000000000036c8cecce8d...,,122207,122207,23032966,0xf00d4b5d,"changeOwner(address _from, address _to)"
2,65342,2015-08-10 19:35:15,0x621de9a006b56c425d21ee0e04ab25866fff4cf606dd...,1,0x889d18b8791f43688d07e0b588e94de746a020d4337c...,0,0x3fb1cd2cd96c6d5c0b5eb3322d807b34482481d4,0xde0b295669a9fd93d5f28d9ec85e40f4cb697bae,0,122269,50000000000,0,,0xf00d4b5d00000000000000000000000005096a47749d...,,122207,122207,23032828,0xf00d4b5d,"changeOwner(address _from, address _to)"
3,68413,2015-08-11 10:04:14,0x1c1fb39d78d2ddfbba1ebd38076ccf54b8aaf3910839...,0,0x822cb48acc3c839264ec9cb247600e05089d72f8aea5...,0,0x063dd253c8da4ea9b12105781c9611b8297f5d14,0xde0b295669a9fd93d5f28d9ec85e40f4cb697bae,0,150000,60137282256,0,,0xf00d4b5d000000000000000000000000036c8cecce8d...,,36733,36733,23029757,0xf00d4b5d,"changeOwner(address _from, address _to)"
4,68425,2015-08-11 10:07:51,0xb8ce46e64f5fbaec38073e592e00bce29f17cbcccc3e...,1,0xaf01ce827008d1a1ee157ddb06866bc7e01a9375d1bd...,0,0x063dd253c8da4ea9b12105781c9611b8297f5d14,0xde0b295669a9fd93d5f28d9ec85e40f4cb697bae,0,150000,60251851460,0,,0xf00d4b5d00000000000000000000000005096a47749d...,,36733,36733,23029745,0xf00d4b5d,"changeOwner(address _from, address _to)"


### Get Event Logs for a specific address

In [ ]:
# Events usually contain information about specific actions that occurred in a smart contract,
# such as token transfers, contract creations, or other significant events.

params = {
    'chainid':1, # Chain ID for Ethereum Mainnet
    'module':'logs', # Module for account-related actions
    'action':'getLogs', # Action to get the account balance
    'address':'0xbd3531da5cf5857e7cfaa92426877b022e612cf8', # contract address
    'startblock':12878196, # Tag to specify the latest block
    'endblock':12878196, # Tag to specify the latest block
    'apikey':ETHERSCAN_API_KEY # API Key for Etherscan
}
r = requests.get(etherscan_url, params=params) # Making the GET request to Etherscan API
r.raise_for_status() # Raise an error for bad responses (4xx or 5xx status codes)
data = r.json() # Parsing the JSON response

In [13]:
# Display the data
print(f'API Response:\n{json.dumps(data, indent=4)}')

API Response:
{
    "status": "1",
    "message": "OK",
    "result": [
        {
            "address": "0xbd3531da5cf5857e7cfaa92426877b022e612cf8",
            "topics": [
                "0x8be0079c531659141344cd1fd0a4f28419497f9722a3daafe3b4186f6b6457e0",
                "0x0000000000000000000000000000000000000000000000000000000000000000",
                "0x000000000000000000000000e9da256a28630efdc637bfd4c65f0887be1aeda8"
            ],
            "data": "0x",
            "blockNumber": "0xc47993",
            "blockHash": "0x13e906e02f1139e192dd23116e06275cc6d5e5a3447cb9e8085b1a041283ff92",
            "timeStamp": "0x60f963d9",
            "gasPrice": "0x4a817c800",
            "gasUsed": "0x415dfd",
            "logIndex": "0x8",
            "transactionHash": "0xe632f3e39c73b57a160c44451ff4ba8f19b80a0e60ab54a4e686a0f0687d81d4",
            "transactionIndex": "0x3"
        },
        {
            "address": "0xbd3531da5cf5857e7cfaa92426877b022e612cf8",
            "topics"

In [15]:
df = pd.DataFrame(data['result']) # Converting the logs to a pandas DataFrame

df.head()

,address,topics,data,blockNumber,blockHash,timeStamp,gasPrice,gasUsed,logIndex,transactionHash,transactionIndex
0,0xbd3531da5cf5857e7cfaa92426877b022e612cf8,[0x8be0079c531659141344cd1fd0a4f28419497f9722a...,0x,0xc47993,0x13e906e02f1139e192dd23116e06275cc6d5e5a3447c...,0x60f963d9,0x4a817c800,0x415dfd,0x8,0xe632f3e39c73b57a160c44451ff4ba8f19b80a0e60ab...,0x3
1,0xbd3531da5cf5857e7cfaa92426877b022e612cf8,[0x62e78cea01bee320cd4e420270b5ea74000d11b0c9f...,0x000000000000000000000000e9da256a28630efdc637...,0xc47993,0x13e906e02f1139e192dd23116e06275cc6d5e5a3447c...,0x60f963d9,0x4a817c800,0x415dfd,0x9,0xe632f3e39c73b57a160c44451ff4ba8f19b80a0e60ab...,0x3
2,0xbd3531da5cf5857e7cfaa92426877b022e612cf8,[0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c...,0x,0xc479f6,0x62d9dff817862d975a478b8cba7c50342f0ec3f29977...,0x60f96922,0x572365f42,0x1f7e3,0xac,0x5de14778174290ec856c6f3bf0aff822a3e91a3c3274...,0x3b
3,0xbd3531da5cf5857e7cfaa92426877b022e612cf8,[0x645f26e653c951cec836533f8fe0616d301c20a1715...,0x,0xc479f6,0x62d9dff817862d975a478b8cba7c50342f0ec3f29977...,0x60f96922,0x572365f42,0x1f7e3,0xad,0x5de14778174290ec856c6f3bf0aff822a3e91a3c3274...,0x3b
4,0xbd3531da5cf5857e7cfaa92426877b022e612cf8,[0x5db9ee0a495bf2e6ff9c91a7834c1ba4fdd244a5e8a...,0x000000000000000000000000e9da256a28630efdc637...,0xc4816e,0x9d510559dba6ba79f2945ff241572d181f90cf64b510...,0x60f9ce26,0x49c2c0600,0x6fbd,0x10c,0xb1614b67923d5fb8e3ab9b4112c39836e3b0b1519555...,0x7f


### Get ERC-20 Token Supply by ContractAddresses

In [16]:
params = {
    'chainid':1, # Chain ID for Ethereum Mainnet
    'module':'stats', # Module for account-related actions
    'action':'tokensupply', # Action to get the token supply
    'contractaddress':'0x6b175474e89094c44da98b954eedeac495271d0f', # Contract address for DAI token
    'apikey':ETHERSCAN_API_KEY # API Key for Etherscan
}

response = requests.get(url=etherscan_url, params=params) # Making a GET request to the Etherscan API with the specified parameters
response.raise_for_status()
data = response.json() # Parsing the JSON response from the API

# Display the data
print(f'API Response:\n{json.dumps(data, indent=4)}')

API Response:
{
    "status": "1",
    "message": "OK",
    "result": "3792764745461665128048293474"
}


In [20]:
# Convert the token supply from Wei to Ether
token_supply_wei = int(data['result']) # Extracting the token supply in Wei from the response
token_supply_ether = token_supply_wei / 10**18 

print(f'Token Supply in Ether: {token_supply_ether:,.2f} DAI') # Displaying the token supply in Ether

Token Supply in Ether: 3,792,764,745.46 DAI


### Get ERC-20 Token Account Balance for Token Contract Address

In [22]:
# Returns the current balance of a specific address for a specific ERC-20 token contract
# This is similar to the previous example, but we are using a different action to get the balance of a specific token for a specific address.

params = {
    'chainid':1, # Chain ID for Ethereum Mainnet
    'module':'account', # Module for account-related actions
    'action':'tokenbalance', # Action to get the token balance
    'contractaddress':'0x6b175474e89094c44da98b954eedeac495271d0f', # Contract address for DAI token
    'address':'0xde0b295669a9fd93d5f28d9ec85e40f4cb697bae', # Address to get the token balance for
    'tag':'latest', # Tag to specify the latest block
    'apikey':ETHERSCAN_API_KEY # API Key for Etherscan
}

try:
    response = requests.get(url=etherscan_url, params=params) # Making a GET request to the Etherscan API with the specified parameters
    response.raise_for_status() # Raises an HTTPError for bad responses (4xx or 5xx status codes)
    data = response.json() # Parsing the JSON response from the API

except requests.exceptions.HTTPError as err:
    print(f'HTTP error occurred: {err}')



In [23]:
# display the data
print(f'API Response:\n{json.dumps(data, indent=4)}') # Displaying the API response in a formatted JSON structure

API Response:
{
    "status": "1",
    "message": "OK",
    "result": "0"
}


### Gas Tracker Metrics

In [24]:
# Gas oracle API provides real-time gas prices for Ethereum transactions.
# Gas Oracle prices (for priority fees)
# The gas prices are returned in Gwei, which is a subunit of Ether (1 Ether = 10^9 Gwei).

params = {
    'chainid':1, # Chain ID for Ethereum Mainnet
    'module':'gastracker', # Module for gas tracker actions
    'action':'gasoracle', # Action to get the gas oracle prices
    'apikey':ETHERSCAN_API_KEY
}

response = requests.get(url=etherscan_url, params=params)
response.raise_for_status() # Raises an HTTPError for bad responses (4xx or 5xx status codes)
data = response.json() # Parsing the JSON response from the API

print(f'API Response:\n{json.dumps(data, indent=4)}') # Displaying the API response in a formatted JSON structure

API Response:
{
    "status": "1",
    "message": "OK",
    "result": {
        "LastBlock": "23099362",
        "SafeGasPrice": "0.403509126",
        "ProposeGasPrice": "0.403532347",
        "FastGasPrice": "0.482509126",
        "suggestBaseFee": "0.403509126",
        "gasUsedRatio": "0.460033980574326,0.232118914623056,0.93550704742256,0.459418422222222,0.528891133333333"
    }
}
